# Timescale decomposition

The raw fluorescence autocorrelation is compared with episode centering, linear detrending, and moving-average high-pass residuals. This tests whether apparent long memory is an intrinsic dynamical timescale or an episode-level baseline/trend.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig

ROOT = Path.cwd()
if ROOT.name == 'analysis' and ROOT.parent.name == 'agent_workspace':
    ROOT = ROOT.parent.parent
INPUT = ROOT / 'agent_workspace/logs/scripts/timescale_decomposition/initial'
OUTPUT = ROOT / 'figures/261027时间尺度分解'
OUTPUT.mkdir(parents=True, exist_ok=True)
acf = pd.read_csv(INPUT / 'autocorrelation_by_preprocessing.csv')
energy = pd.read_csv(INPUT / 'episode_energy_fractions.csv')
acf.head()

,state,method,lag_steps,seconds,mean,median,p10,p90,n_pairs
0,Awake,raw,1,0.241178,0.844598,0.881324,0.660452,0.967786,17058
1,Awake,raw,2,0.482356,0.808576,0.841670,0.612259,0.956064,17011
2,Awake,raw,4,0.964712,0.749898,0.775545,0.529090,0.939562,16917
3,Awake,raw,8,1.929425,0.668052,0.678045,0.402955,0.918671,16729
4,Awake,raw,16,3.858850,0.574959,0.557578,0.257101,0.899489,16353


In [2]:
states = ['Awake', 'NREM', 'REM']
methods = ['raw', 'episode_centered', 'linear_detrended', 'highpass_64s', 'highpass_16s']
labels = {'raw': 'Raw', 'episode_centered': 'Episode-centered', 'linear_detrended': 'Linear detrend', 'highpass_64s': '64 s residual', 'highpass_16s': '16 s residual'}
colors = {'raw': '#000000', 'episode_centered': '#D55E00', 'linear_detrended': '#CC79A7', 'highpass_64s': '#0072B2', 'highpass_16s': '#009E73'}
fig, axes = get_fig(RN=1, CN=3, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(np.asarray(axes).reshape(-1))
for state in states:
    ax = next(ax_loader)
    part = acf[acf.state == state]
    for method in methods:
        group = part[part.method == method].sort_values('seconds')
        ax.plot(group.seconds, group['median'], marker='o', ms=3, color=colors[method], label=labels[method])
    ax.axhline(0, color='black', ls='--', lw=0.8)
    ax.set_xscale('log')
    ax.set_xlabel('Lag (s)')
    ax.set_ylabel('Median autocorrelation')
    ax.set_title(state)
    if state == 'Awake':
        ax.legend(frameon=False, fontsize=8)
for suffix in ('png', 'svg', 'pdf'):
    fig.savefig(OUTPUT / f'autocorrelation_preprocessing.{suffix}', bbox_inches='tight')
plt.show()

In [3]:
table = acf[acf.lag_steps.isin([1, 8, 32, 128, 256])].pivot_table(index=['state', 'method'], columns='seconds', values='median')
table.round(3)

seconds                 0.241178   1.929425   7.717699   30.870796  61.741592
state method                                                                 
Awake episode_centered      0.711      0.365      0.063     -0.002     -0.004
      highpass_16s          0.588      0.087     -0.138      0.001      0.005
      highpass_64s          0.684      0.298     -0.023     -0.053      0.000
      linear_detrended      0.687      0.307      0.016     -0.021     -0.014
      raw                   0.881      0.678      0.458      0.418      0.408
NREM  episode_centered      0.657      0.258     -0.035     -0.016     -0.008
      highpass_16s          0.577      0.086     -0.148      0.001      0.016
      highpass_64s          0.644      0.236     -0.070     -0.028      0.018
      linear_detrended      0.629      0.212     -0.066     -0.020      0.014
      raw                   0.836      0.575      0.309      0.273      0.215
REM   episode_centered      0.797      0.491      0.058      0.001     -0.072
      highpass_16s          0.662      0.171     -0.193      0.006      0.011
      highpass_64s          0.774      0.433     -0.029     -0.018     -0.029
      linear_detrended      0.779      0.447     -0.008     -0.013     -0.049
      raw                   0.880      0.638      0.293      0.206      0.115

## Interpretation

Raw traces retain substantial correlation at 32–64 s. Episode centering or linear detrending drives the median correlation close to zero by about 8 s in all states. Thus most apparent ultra-slow memory is attributable to episode offsets and slow drift. The remaining sub-second to several-second correlation may mix population dynamics, calcium-indicator kinetics, exposure integration, and other measurement filtering; it is not yet an intrinsic neuronal time constant.